# Step 2 — Prompt Engineering for Conversation Intelligence
**Tough Talks · Phase 1**

Goal: validate the four prompt families that drive the rest of the system, with each output structured as JSON matching its schema in `data/schemas/`.

1. **Adversarial persona** — multi-turn (3 turns) — sampled — output: `persona_reply.schema.json`
2. **Debrief** — single-turn — greedy — output: `debrief.schema.json`
3. **Pre-mortem** — single-turn — greedy — output: `premortem.schema.json`
4. **Cultural calibration** — single-turn — greedy — output: `cultural.schema.json`

Prompts live in `data/prompts/<name>.md` (markdown with `$placeholder` slots). Output schemas live in `data/schemas/<name>.schema.json`. Both are loaded at notebook startup; cells just render and execute. All non-trivial logic stays in `backend/core/_runtime/`.

In [ ]:
# ── 0. Install / upgrade dependencies ────────────────────────────────────────
# Same as Step 1: only bump transformers + accelerate. Do NOT bump torch on
# Colab/Kaggle (it breaks the torchvision pairing). Restart the kernel after
# this cell finishes the FIRST time you run it in a session.

!pip install -q -U transformers accelerate

In [ ]:
# ── 1. Locate (or fetch) the repo, put it on sys.path ───────────────────────
# Tries cwd ancestors and known cloud workspace dirs first. If no clone is
# present, clones REPO_URL into the workspace. If a clone exists, refreshes
# it from origin so the cloud kernel always runs the latest pushed code.

import os, pathlib, subprocess, sys

REPO_URL  = "https://github.com/EhsanFarazmand/tough_talks.git"
REPO_NAME = "tough_talks"

def _looks_like_repo(p: pathlib.Path) -> bool:
    return (p / "backend" / "core" / "_runtime").is_dir()

def _scan_for_repo() -> pathlib.Path | None:
    cwd = pathlib.Path.cwd()
    for parent in [cwd, *cwd.parents]:
        if _looks_like_repo(parent):
            return parent
    for base in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")):
        candidate = base / REPO_NAME
        if _looks_like_repo(candidate):
            return candidate
    return None

def _refresh(target: pathlib.Path) -> None:
    if not (target / ".git").is_dir():
        return
    print(f"Refreshing {target} from origin")
    subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1", "origin"], capture_output=True, check=False)
    subprocess.run(["git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"], capture_output=True, check=False)

REPO_ROOT = _scan_for_repo()
if REPO_ROOT is None:
    base = next((b for b in (pathlib.Path("/content"), pathlib.Path("/kaggle/working")) if b.is_dir()), pathlib.Path.cwd())
    target = base / REPO_NAME
    print(f"Cloning {REPO_URL} -> {target}")
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{result.stderr}")
    REPO_ROOT = target
else:
    _refresh(REPO_ROOT)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

In [ ]:
# ── 2. Imports ───────────────────────────────────────────────────────────────
import json
from dataclasses import dataclass, field
from typing import Any

import torch

from backend.core._runtime import (
    DEFAULT_MODEL_ID,
    GenerationConfig,
    JsonParseError,
    LoadConfig,
    chat,
    load_model,
    load_prompt,
    parse_json,
)

In [ ]:
# ── 3. Configuration ─────────────────────────────────────────────────────────
MODEL_ID = DEFAULT_MODEL_ID                              # google/gemma-4-E2B-it
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

# Greedy default for analytical prompts. Persona uses do_sample=True per-call.
GEN_GREEDY  = GenerationConfig(max_new_tokens=700, do_sample=False)
GEN_PERSONA = GenerationConfig(max_new_tokens=384, do_sample=True)

# Reproducible sampling seed for the persona multi-turn test.
PERSONA_SEED = 42

print(f"Model  : {MODEL_ID}")
print(f"Device : {DEVICE}")

In [ ]:
# ── 4. Load processor + model ────────────────────────────────────────────────
# Step 2 is text-only → AutoModelForCausalLM (default).

processor, model = load_model(LoadConfig(model_id=MODEL_ID))
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded ({n_params:.1f}B parameters, on {model.device})")

In [ ]:
# ── 5. Load prompt templates ────────────────────────────────────────────────
# Each prompt's required placeholders are derived from its $name slots
# at load time. Render-time mismatches raise PromptError loudly.

PROMPTS = {name: load_prompt(name) for name in ["persona", "debrief", "premortem", "cultural"]}

for name, p in PROMPTS.items():
    print(f"  - {name:10s}  placeholders: {sorted(p.placeholders)}")

In [ ]:
# ── 6. Helpers ───────────────────────────────────────────────────────────────
# generate_json() wraps the chat → parse_response → parse_json pipeline so
# every prompt test goes through the same path.
# _check_keys() does a lightweight schema check (required keys + types only).
# Heavy schema validation is left to scripts/validate_schemas.py and CI.

@dataclass
class TestResult:
    name: str
    ok: bool = False
    note: str = ""
    parsed: Any = None
    raw: str = ""
    failures: list[str] = field(default_factory=list)

RESULTS: list[TestResult] = []

def generate_json(messages: list[dict], cfg: GenerationConfig) -> tuple[Any, str]:
    """Run chat → parse_response → parse_json. Returns (parsed, raw)."""
    raw = chat(processor, model, messages=messages, cfg=cfg)
    text = processor.parse_response(raw)
    if not isinstance(text, str):
        # If parse_response returned a dict/object, look for the text content.
        if isinstance(text, dict):
            text = text.get("content") or text.get("text") or json.dumps(text)
        else:
            text = getattr(text, "text", None) or str(text)
    parsed = parse_json(text)
    return parsed, raw

def _check_keys(parsed: Any, required: list[str]) -> list[str]:
    if not isinstance(parsed, dict):
        return [f"top-level not a dict: {type(parsed).__name__}"]
    return [f"missing key: {k}" for k in required if k not in parsed]

In [ ]:
# ── 7. Adversarial persona — multi-turn (3 turns) ───────────────────────────
# Scenario: salary negotiation. David, the manager, deflects with budget
# language. The user pushes 3 times. Persona must stay in character across
# turns and emit valid JSON every time. Each turn's full JSON output is fed
# back as the assistant's prior message so the model sees its own state.

torch.manual_seed(PERSONA_SEED)

persona_system = PROMPTS["persona"].render(
    persona_name="David",
    profile=(
        "Deflects with budget language when cornered. "
        "Hates being surprised in conversations. "
        "Uses 'we' language to dilute personal accountability. "
        "Responds better to data than emotion."
    ),
    user_goal="Ask for a 15% salary raise with market data as evidence.",
)

USER_TURNS = [
    "David, I've been researching market rates and someone with my experience and skills earns about 15% more at comparable companies. I'd like to discuss adjusting my salary to match that.",
    "I appreciate the budget context, but the data I'm bringing isn't a guess — it's three concrete sources showing the gap. What would it take, on the data side, to make this conversation real?",
    "I hear that we're in a tight cycle. Given the gap is documented and I'm asking for parity not a premium, can we agree on a specific number and a date by which we'll revisit it?",
]

history: list[dict] = [{"role": "system", "content": persona_system}]
persona_turns: list[dict] = []
result = TestResult(name="persona_multiturn")

for n, user_msg in enumerate(USER_TURNS, start=1):
    history.append({"role": "user", "content": user_msg})
    try:
        parsed, raw = generate_json(history, GEN_PERSONA)
    except JsonParseError as exc:
        result.failures.append(f"turn {n}: parse failed — {exc}")
        result.raw = exc.raw
        break
    issues = _check_keys(parsed, ["persona_name", "reply", "resistance_type", "escalation_level"])
    if not issues:
        rt = parsed["resistance_type"]
        if rt not in {"deflect", "guilt_trip", "deny", "counter_attack", "silent", "concede"}:
            issues.append(f"resistance_type out of enum: {rt!r}")
        try:
            esc = float(parsed["escalation_level"])
            if not 0.0 <= esc <= 1.0:
                issues.append(f"escalation_level out of range: {esc}")
        except (TypeError, ValueError):
            issues.append(f"escalation_level not numeric: {parsed['escalation_level']!r}")
    if issues:
        result.failures.append(f"turn {n}: {', '.join(issues)}")
        result.parsed = parsed
        break

    persona_turns.append(parsed)
    print(f"--- Turn {n} ---")
    print(f"USER: {user_msg}")
    print(f"PERSONA: {parsed['reply']}")
    print(f"  resistance={parsed['resistance_type']}, escalation={parsed['escalation_level']}")
    print()

    # Feed the JSON back so the model sees its own structured state next turn.
    history.append({"role": "assistant", "content": json.dumps(parsed)})

result.parsed = persona_turns
result.ok = (not result.failures) and len(persona_turns) == len(USER_TURNS)
result.note = (
    f"{len(persona_turns)}/{len(USER_TURNS)} turns parsed"
    + (f" — {result.failures[0]}" if result.failures else "")
)
RESULTS.append(result)

In [ ]:
# ── 8. Debrief — single-turn ────────────────────────────────────────────────

SAMPLE_TRANSCRIPT = """\
User (turn 1): David, I've been looking at market rates and I think there might be a gap...
David: We appreciate everything you do. Budget is tight right now for the whole team.
User (turn 2): Oh, I totally understand, I'm sorry to bring it up at a bad time — it's just something I've been thinking about.
David: We'll keep it in mind for the annual review cycle.
User (turn 3): Right, yeah, that makes sense. I just feel like maybe we could talk about it then? Sorry.
David: Sure, we'll loop back. Is there anything else?
User (turn 4): No, no, that's fine. Thank you for your time.
"""

debrief_system = PROMPTS["debrief"].render(
    user_goal="Ask for a 15% raise with market data as evidence.",
    transcript=SAMPLE_TRANSCRIPT,
)

result = TestResult(name="debrief")
try:
    parsed, raw = generate_json(
        [{"role": "system", "content": debrief_system},
         {"role": "user",   "content": "Produce the debrief now."}],
        GEN_GREEDY,
    )
    result.parsed = parsed
    result.raw = raw
    issues = _check_keys(parsed, ["ground_lost", "over_apologies", "missed_openings", "wins", "one_fix_next_time"])
    if not issues:
        if not isinstance(parsed["one_fix_next_time"], str) or not parsed["one_fix_next_time"].strip():
            issues.append("one_fix_next_time empty or non-string")
        for arr_key in ("ground_lost", "over_apologies", "missed_openings", "wins"):
            if not isinstance(parsed[arr_key], list):
                issues.append(f"{arr_key} is not a list")
        for item in parsed.get("missed_openings", []):
            if isinstance(item, dict) and "better_line" not in item:
                issues.append("missed_openings item missing 'better_line'")
                break
    result.failures = issues
    result.ok = not issues
    result.note = "ok" if result.ok else "; ".join(issues)
    print(json.dumps(parsed, indent=2))
except JsonParseError as exc:
    result.failures = [f"parse failed: {exc}"]
    result.note = result.failures[0]
    result.raw = exc.raw

RESULTS.append(result)

In [ ]:
# ── 9. Pre-mortem — single-turn ─────────────────────────────────────────────

premortem_system = PROMPTS["premortem"].render(
    conversation_description=(
        "I need to tell my co-founder that I think we should shut down the startup. "
        "He has invested 3 years and his savings. I am the CEO. "
        "We have a board meeting in 2 weeks. "
        "My goal is to reach a mutual decision to wind down gracefully, "
        "preserve the friendship, and agree on the shutdown timeline."
    )
)

result = TestResult(name="premortem")
try:
    parsed, raw = generate_json(
        [{"role": "system", "content": premortem_system},
         {"role": "user",   "content": "Generate the 3 failure scenarios now."}],
        GEN_GREEDY,
    )
    result.parsed = parsed
    result.raw = raw
    issues = _check_keys(parsed, ["goal", "failure_scenarios"])
    if not issues:
        scenarios = parsed.get("failure_scenarios")
        if not isinstance(scenarios, list):
            issues.append("failure_scenarios is not a list")
        elif len(scenarios) != 3:
            issues.append(f"expected 3 scenarios, got {len(scenarios)}")
        else:
            for i, s in enumerate(scenarios, start=1):
                missing = [k for k in ("scenario_id", "title", "description", "likely_trigger", "destabilization_risk", "simulation_parameters") if k not in s]
                if missing:
                    issues.append(f"scenario {i} missing: {missing}")
                    continue
                sim = s.get("simulation_parameters", {})
                if not sim.get("opening_move"):
                    issues.append(f"scenario {i} missing opening_move")
    result.failures = issues
    result.ok = not issues
    result.note = "ok" if result.ok else "; ".join(issues)
    print(json.dumps(parsed, indent=2))
except JsonParseError as exc:
    result.failures = [f"parse failed: {exc}"]
    result.note = result.failures[0]
    result.raw = exc.raw

RESULTS.append(result)

In [ ]:
# ── 10. Cultural calibration — single-turn ──────────────────────────────────

cultural_system = PROMPTS["cultural"].render(
    culture="Japan",
    industry="Software",
    power_dynamic="junior-to-senior, hierarchical",
    conversation_goal=(
        "Tell my senior manager (senpai) that I disagree with a technical "
        "decision he has publicly committed to."
    ),
)

result = TestResult(name="cultural")
try:
    parsed, raw = generate_json(
        [{"role": "system", "content": cultural_system},
         {"role": "user",   "content": "Produce the calibration parameters now."}],
        GEN_GREEDY,
    )
    result.parsed = parsed
    result.raw = raw
    issues = _check_keys(parsed, ["context", "communication_adjustments", "simulation_instructions", "opening_line_recommendations"])
    if not issues:
        adj = parsed.get("communication_adjustments", {})
        try:
            d = float(adj.get("directness_level", -1))
            if not 0.0 <= d <= 1.0:
                issues.append(f"directness_level out of range: {d}")
        except (TypeError, ValueError):
            issues.append("directness_level not numeric")
        if not isinstance(adj.get("face_saving_required"), bool):
            issues.append("face_saving_required is not a bool")
        opens = parsed.get("opening_line_recommendations")
        if not isinstance(opens, list) or len(opens) != 3:
            issues.append(f"opening_line_recommendations should be 3 items, got {opens!r}")
    result.failures = issues
    result.ok = not issues
    result.note = "ok" if result.ok else "; ".join(issues)
    print(json.dumps(parsed, indent=2))
except JsonParseError as exc:
    result.failures = [f"parse failed: {exc}"]
    result.note = result.failures[0]
    result.raw = exc.raw

RESULTS.append(result)

In [ ]:
# ── 11. Step 2 results table ────────────────────────────────────────────────
# Renders unconditionally. For any FAIL row, the failures list and the
# first 500 chars of the raw model output are shown for debugging.

print("=" * 72)
print("STEP 2 RESULTS")
print("=" * 72)
all_ok = True
for r in RESULTS:
    icon = "PASS" if r.ok else "FAIL"
    print(f"[{icon}]  {r.name:20s}  {r.note}")
    if not r.ok:
        all_ok = False

print()
for r in RESULTS:
    if r.ok:
        continue
    print("-" * 72)
    print(f"{r.name} — failures:")
    for f in r.failures:
        print(f"  - {f}")
    if r.raw:
        print(f"raw output (first 500 chars):\n{r.raw[:500]}")

print()
print("OVERALL:", "PHASE 1 COMPLETE — READY FOR PHASE 2" if all_ok else "FIX FAILURES ABOVE")